# imports

In [31]:
import pandas as pd
import numpy as np
from sklearn.metrics import classification_report, accuracy_score
import mlflow
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
import mlflow.tensorflow
from tensorflow.keras.callbacks import EarlyStopping

# config

In [32]:
ASSET = "BTC"
INTERVAL = "1h"

In [33]:
TIMESTEPS = 24

# mlflow

In [34]:
mlflow.set_tracking_uri("http://localhost:5000")

experiment_name = f"{ASSET}_{INTERVAL}_deep_learning"
mlflow.set_experiment(experiment_name)

<Experiment: artifact_location='/mlruns/13', creation_time=1778414378831, experiment_id='13', last_update_time=1778414378831, lifecycle_stage='active', name='BTC_1h_deep_learning', tags={}, trace_location=None, workspace='default'>

# load data

In [35]:
train_df = pd.read_parquet('../../../data/processed/train_btc_1h.parquet')
test_df = pd.read_parquet('../../../data/processed/test_btc_1h.parquet')

# splitting

In [36]:
features = [col for col in train_df.columns if col not in ['target_1h', 'target_direction']]
X_train = train_df[features]
y_train = train_df['target_direction']
X_test = test_df[features]
y_test = test_df['target_direction']

# checking the train data if scaled or not

In [37]:
print(X_train.describe().loc[['mean', 'std']].T.head(5))

                 mean       std
rsi_14   5.919112e-16  1.000012
roc_10   9.638083e-18  1.000012
roc_20  -2.127025e-17  1.000012
stoch_k -4.254050e-16  1.000012
stoch_d  1.661738e-17  1.000012


# LSTM

In [38]:
def create_sequences(X, y, time_steps=24):
    """
    Transforms 2D tabular data into 3D tensors for LSTM consumption.
    X: Numpy array of features
    y: Target series
    time_steps: historical periods to look back
    """
    Xs, ys = [], []
    for i in range(len(X) - time_steps):
        Xs.append(X[i:(i + time_steps)])
        ys.append(y.iloc[i + time_steps])
        
    return np.array(Xs), np.array(ys)


X_train_seq, y_train_seq = create_sequences(X_train.values, y_train, TIMESTEPS)
X_test_seq, y_test_seq = create_sequences(X_test.values, y_test, TIMESTEPS)

print(f"Training Shape (Samples, TimeSteps, Features): {X_train_seq.shape}")
print(f"Testing Shape: {X_test_seq.shape}")

Training Shape (Samples, TimeSteps, Features): (42735, 24, 29)
Testing Shape: (10666, 24, 29)


# model

In [39]:
model = Sequential([
    LSTM(64, return_sequences=True, input_shape=(TIMESTEPS, X_train_seq.shape[2])),
    Dropout(0.2),
    
    LSTM(32, return_sequences=False),
    Dropout(0.2),
    
    Dense(16, activation='relu'),
    
    Dense(1, activation='sigmoid')
])

c:\Users\Manindra\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\rnn\rnn.py:204: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


# compile model

In [40]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [41]:
model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_4 (LSTM)                   │ (None, 24, 64)         │        24,064 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 24, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_5 (LSTM)                   │ (None, 32)             │        12,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 37,025 (144.63 KB)

 Trainable params: 37,025 (144.63 KB)

 Non-trainable params: 0 (0.00 B)

# mlflow tensorflow 

In [42]:
mlflow.tensorflow.autolog()

2026/05/10 14:57:04 WARNING mlflow.utils.autologging_utils: MLflow tensorflow autologging is known to be compatible with 2.16.2 <= tensorflow, but the installed version is 2.16.1. If you encounter errors during autologging, try upgrading / downgrading tensorflow to a compatible version, or try upgrading MLflow.


# early stopping

In [43]:
early_stopping = EarlyStopping(
    monitor='val_loss', 
    patience=5,
    restore_best_weights=True
)

# run, fit model with mlflow

In [44]:
with mlflow.start_run(run_name="LSTM_Baseline_run4"):
    
    mlflow.log_param("time_steps", TIMESTEPS)
    mlflow.log_param("model_type", "LSTM_2_Layer")
    
    print("started training")
    
    history = model.fit(
        X_train_seq, y_train_seq,
        epochs=50,
        batch_size=64,
        validation_split=0.2,
        callbacks=[early_stopping],
        verbose=1
    )
    
    test_loss, test_acc = model.evaluate(X_test_seq, y_test_seq, verbose=0)
    
    mlflow.log_metric("test_loss", test_loss)
    mlflow.log_metric("test_accuracy", test_acc)
    mlflow.keras.log_model(model, "lstm_model")
    
    print(f"\n--- Training Complete ---")
    print(f"Test Accuracy: {test_acc:.4f}")

started training


Epoch 1/50
534/535 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.5022 - loss: 0.6951

535/535 ━━━━━━━━━━━━━━━━━━━━ 11s 15ms/step - accuracy: 0.5022 - loss: 0.6951 - val_accuracy: 0.5109 - val_loss: 0.6934
Epoch 2/50
534/535 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.5097 - loss: 0.6928

535/535 ━━━━━━━━━━━━━━━━━━━━ 8s 15ms/step - accuracy: 0.5097 - loss: 0.6928 - val_accuracy: 0.5106 - val_loss: 0.6928
Epoch 3/50
535/535 ━━━━━━━━━━━━━━━━━━━━ 8s 15ms/step - accuracy: 0.5121 - loss: 0.6926 - val_accuracy: 0.5123 - val_loss: 0.6929
Epoch 4/50
535/535 ━━━━━━━━━━━━━━━━━━━━ 8s 15ms/step - accuracy: 0.5245 - loss: 0.6914 - val_accuracy: 0.5139 - val_loss: 0.6934
Epoch 5/50
532/535 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.5314 - loss: 0.6910

535/535 ━━━━━━━━━━━━━━━━━━━━ 8s 15ms/step - accuracy: 0.5313 - loss: 0.6910 - val_accuracy: 0.5166 - val_loss: 0.6923
Epoch 6/50
535/535 ━━━━━━━━━━━━━━━━━━━━ 8s 15ms/step - accuracy: 0.5324 - loss: 0.6904 - val_accuracy: 0.5195 - val_loss: 0.6924
Epoch 7/50
535/535 ━━━━━━━━━━━━━━━━━━━━ 8s 15ms/step - accuracy: 0.5357 - loss: 0.6903 - val_accuracy: 0.5194 - val_loss: 0.6927
Epoch 8/50
535/535 ━━━━━━━━━━━━━━━━━━━━ 8s 15ms/step - accuracy: 0.5294 - loss: 0.6907 - val_accuracy: 0.5146 - val_loss: 0.6928
Epoch 9/50
535/535 ━━━━━━━━━━━━━━━━━━━━ 8s 15ms/step - accuracy: 0.5390 - loss: 0.6886 - val_accuracy: 0.5150 - val_loss: 0.6927
Epoch 10/50
535/535 ━━━━━━━━━━━━━━━━━━━━ 8s 15ms/step - accuracy: 0.5351 - loss: 0.6891 - val_accuracy: 0.5141 - val_loss: 0.6932
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 241ms/step


2026/05/10 14:58:28 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/10 14:58:39 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/10 14:58:40 WARNING mlflow.keras.save: You are saving a Keras model without specifying model signature.



--- Training Complete ---
Test Accuracy: 0.5212
🏃 View run LSTM_Baseline_run4 at: http://localhost:5000/#/experiments/13/runs/534206b53f694aa991e6bc07702031b1
🧪 View experiment at: http://localhost:5000/#/experiments/13
